In [1]:
import sympy as sp
from sympy.solvers.ode.systems import dsolve_system
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Math, display
sp.init_printing()

In [2]:
def fancy_display(*args, sep=' ', latexer=sp.latex, mather=Math):
    """
    Принимает произвольное количество строк и объектов SymPy.
    Строки добавляет как есть, остальные объекты переводит в LaTeX.
    """
    if mather is None:
        mather = lambda i: i
    if latexer is None:
        latexer = lambda i: str(i)

    parts = []
    for item in args:
        if isinstance(item, str):
            parts.append(item)
        else:
            parts.append(latexer(item))

    full_latex = sep.join(parts)
    display(mather(full_latex))

## Задание 1

> Для поверхности вращения
  $$ \bar{r}(u, v) = (f(u) \cos v, f(u) \sin v, g(u)) $$
  найти все символы Кристоффеля первой квадратичной формы.

In [3]:
u1, u2 = sp.symbols('u^1,u^2', real=True)

# Объявляем f и g как неизвестные функции
f = sp.Function('f')
g = sp.Function('g')

In [4]:
r = sp.Matrix([
    f(u1) * sp.cos(u2),
    f(u1) * sp.sin(u2),
    g(u1)
])

coords = [u1, u2]

fancy_display(r'r = ', r, r'.')

<IPython.core.display.Math object>

In [5]:
def get_normal(r, coords):
    """
    Вычисляет нормаль к поверхности в точке (u, v)
    """
    
    u, v = coords
    
    r_u = r.diff(u)
    r_v = r.diff(v)
    
    n = r_u.cross(r_v)
    n /= n.norm()
    return n

def get_I_and_II(r, coords):
    """
    Вычисляет первую и вторую квадратичную формы
    Возвращает коэффициенты этих форм
    """
    
    u, v = coords
    
    r_u = r.diff(u)
    r_v = r.diff(v)
    
    n = get_normal(r, coords)
    
    r_uu = r_u.diff(u)
    r_uv = r_u.diff(v)
    r_vv = r_v.diff(v)

    E = r_u.dot(r_u).simplify()
    F = r_u.dot(r_v).simplify()
    G = r_v.dot(r_v).simplify()
    
    L = r_uu.dot(n).simplify()
    M = r_uv.dot(n).simplify()
    N = r_vv.dot(n).simplify()

    I  = (E, F, G)
    II = (L, M, N)

    return I, II

In [6]:
def get_metric_tensor(r, coords):
    """
    Вычисляет метрический тензор
    """
    
    (E, F, G), _ = get_I_and_II(r, coords)
    G = sp.Matrix([[E, F], [F, G]])
    
    return G

In [7]:
def get_cristoffel_symbols(r, coords):
    """
    Вычисляет метрический тензор
    """

    dim = 2 

    G = get_metric_tensor(r, coords)
    G_inv = G.inv() # обратный к метрическому тензор
    
    # Создаем пустой 3D-массив (тензор типа (2, 1))
    names = sp.MutableDenseNDimArray.zeros(dim, dim, dim)
    cristoffel_symbols = sp.MutableDenseNDimArray.zeros(dim, dim, dim)
    
    # Заполняем символами с верхним и нижними индексами (нумерация с 1: 1, 2)
    for k in range(dim):
        for i in range(dim):
            for j in range(dim):
                # Формируем LaTeX-имя: \Gamma^{k}_{i j}
                latex_name = rf'\Gamma^{{{k+1}}}_{{{i+1}{j+1}}}'
                names[k, i, j] = sp.Symbol(latex_name)
                
                # Непосредственно символы Кристоффеля
                value = sp.Rational(1, 2) * sum(
                    G_inv[k, l] * (
                        G[j, l].diff(coords[i]) + G[i, l].diff(coords[j]) - G[i, j].diff(coords[l])
                    )
                    for l in range(dim)
                )
                cristoffel_symbols[k, i, j] = sp.simplify(value)

    
    return cristoffel_symbols, names

def display_cristoffel_symbols(cristoffel_symbols, cristoffel_names):
    """
    Выводит символы Кристоффеля
    """
    
    dim = 2
    for k in range(dim):
        for i in range(dim):
            for j in range(dim):
                fancy_display(cristoffel_names[k, i, j], r' = ', cristoffel_symbols[k, i, j])

In [8]:
cristoffel_symbols, cristoffel_names = get_cristoffel_symbols(r, coords)

display_cristoffel_symbols(cristoffel_symbols, cristoffel_names)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Задание 2

> Для сферы радиуса $R$ рассмотреть экватор
  $$ \theta = \frac{\pi}{2}, \quad \varphi = t. $$
  Записать уравнения параллельного переноса касательного векторного поля вдоль экватора и решить их для произвольного начального вектора.

In [9]:
theta, phi, t = sp.symbols(r'\theta, \varphi, t', real=True)
R = sp.Symbol('R', real=True, positive=True)

In [10]:
r = sp.Matrix([
    R * sp.sin(theta) * sp.cos(phi),
    R * sp.sin(theta) * sp.sin(phi),
    R * sp.cos(theta)
])
curve_parametrization = sp.Matrix([sp.pi/2, t])
param = {theta: curve_parametrization[0], phi: curve_parametrization[1]}
gamma = r.subs(param)

coords = [theta, phi]

fancy_display(r'r(\varphi, \theta) = ', r, r',')
fancy_display(r'\gamma(t) = ', gamma, r'.')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [11]:
cristoffel_symbols, cristoffel_names = get_cristoffel_symbols(r, coords)
display_cristoffel_symbols(cristoffel_symbols, cristoffel_names)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [12]:
x1, x2 = sp.symbols("x^1, x^2", cls=sp.Function)
x = [x1(t), x2(t)]
curve_parametrization_dots = curve_parametrization.diff(t)

equations = [0, 0]
dim = 2

for k in range(dim):
    for i in range(dim):
        for j in range(dim):
            right_part = \
                -sum(
                    cristoffel_symbols[k, i, j] * curve_parametrization_dots[i] * x[j]
                    for j in range(dim)
                    for i in range(dim)
                )
            right_part = right_part.subs(param)
            left_part = x[k].diff(t)
            
            equations[k] = sp.Eq(left_part, right_part)

for k in range(dim):
    fancy_display(equations[k])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [13]:
# Решаем систему аналитически
solutions = dsolve_system(equations, funcs=x, t=t)

# Выводим полученное общее решение через ваш fancy_display
for sol in solutions:
    fancy_display(sol)

<IPython.core.display.Math object>

In [14]:
def get_parallel_shift(r, gamma, curve_parametrization, coords, t):
    cristoffel_symbols, _ = get_cristoffel_symbols(r, coords)

    x1, x2 = sp.symbols("x^1, x^2", cls=sp.Function)
    x = [x1(t), x2(t)]
    curve_parametrization_dots = curve_parametrization.diff(t)
    
    equations = [0, 0]
    dim = 2
    
    for k in range(dim):
        for i in range(dim):
            for j in range(dim):
                right_part = \
                    -sum(
                        cristoffel_symbols[k, i, j] * curve_parametrization_dots[i] * x[j]
                        for j in range(dim)
                        for i in range(dim)
                    )
                right_part = right_part.subs(param)
                left_part = x[k].diff(t)
                
                equations[k] = sp.Eq(left_part, right_part)
    
    solutions = dsolve_system(equations, funcs=x, t=t)

    return equations, solutions

## Задание 3

> На цилиндре
  $$ \bar{r}(u, v) = (R \cos v, R \sin v, u) $$
  найти параллельный перенос произвольного касательного вектора вдоль винтовой линии
  $$ u = a t, \quad v = t. $$
  Сравнить изменение его координатных компонент с изменением самого вектора в $\mathbb{R}^3$.

In [15]:
u, v, t = sp.symbols('u, v, t', real=True)
R, a = sp.symbols('R, a', real=True, positive=True)

In [16]:
r = sp.Matrix([
    R * sp.cos(v),
    R * sp.sin(v),
    u
])

coords = [u, v]

curve_parametrization = sp.Matrix([a * t, t])
param = {u: curve_parametrization[0], v: curve_parametrization[1]}
gamma = r.subs(param)


fancy_display(r'r = ', r, r',')
fancy_display(r'\gamma(t) = ', gamma, r'.')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [17]:
equations, solutions = get_parallel_shift(r, gamma, curve_parametrization, coords, t)
solutions = solutions[0]

fancy_display(equations)
fancy_display(solutions)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Координатные компоненты не меняются, в отличие от вектора:

In [18]:
fancy_display(r'r(\gamma(t)) = ', gamma.diff(t))

<IPython.core.display.Math object>

## Задание 4

> Пусть $\gamma(s)$ — кривая на поверхности, параметризованная натуральным параметром, а её единичный касательный вектор $\bar{\tau} = \dot{\gamma}$ параллелен вдоль $\gamma$:
  $$ \nabla_{\bar{\tau}} \bar{\tau} = 0. $$
  Докажите, что вектор ускорения $\ddot{\gamma}$ во всех точках перпендикулярен поверхности.

$\nabla_\tau$ — линейный оператор в евклидовом пространстве, а значит, равен скалярному произведению с фиксированным вектором $h$:
$$
\nabla_\tau = (h, \cdot)
$$

Поскольку $\tau \in \ker \nabla_\tau$, то есть
$$
\nabla_\tau \tau = (h, \tau) = 0,
$$
вектор $h$ лежит в нормальной плоскости $\tau_\perp$ к кривой $\gamma(s)$.

По условию предполагается регулярность кривой $\gamma$, значит, в частности, вектор $\ddot{\gamma}$ будет лежать в ортогональном дополнении $\tau_\perp$, то есть будет ортогонален $\tau = \dot{\gamma}$.

Что и требовалось доказать.